# 04 · Robustness, errors and ablations
`metrics.json` (robustness), `errors.json` and `ablations.json` (Δ vs the full system with a paired group bootstrap, 90 % interval).

In [1]:
import json
from pathlib import Path

import pandas as pd

R = Path("../reports")  # the only directory read (scripts/run_notebooks.py --check-reads-only)


def load(name):
    return json.loads((R / name).read_text(encoding="utf-8"))
m = load("metrics.json")
print(json.dumps(m.get("robustness", {}), indent=1)[:1500])

{
 "note": "data/robustness/probes.csv does not exist yet (P2.5 is a human task); per-style scores appear once it is written",
 "status": "skipped"
}


In [2]:
err = load("errors.json")
e = pd.DataFrame(err["our_test"])
if len(e):
    e["gold_cat"] = e.gold.str.split(" / ").str[0]
    e["pred_cat"] = e.pred.str.split(" / ").str[0]
    display(e.groupby(["gold_cat", "pred_cat", "lang"]).size().sort_values(ascending=False).head(15))

gold_cat          pred_cat          lang   
Finance           Registration      ar         7
                                    arabizi    5
Transportation    Transportation    en         5
                                    arabizi    5
Academic Support  Facilities        ar         4
Transportation    Transportation    ar         4
Registration      Finance           en         4
Facilities        Facilities        ar         4
Academic Support  IT Support        en         4
Transportation    Facilities        ar         3
Facilities        Facilities        en         2
Registration      Registration      en         2
Academic Support  Registration      arabizi    2
                                    ar         2
Registration      Academic Support  en         2
dtype: int64

In [3]:
abl = load("ablations.json")
rows = []
for arm, a in abl["arms"].items():
    for target in ("category", "priority"):
        o = a.get("oof_official", {}).get(target, {})
        t = a.get("our_test", {}).get(target, {})
        rows.append({"arm": arm, "target": target, "oof": o.get("macro_f1"), "oof_delta": o.get("delta"),
                     "oof_ci90": o.get("ci90"), "our_test": t.get("macro_f1"),
                     "our_test_delta": t.get("delta")})
display(pd.DataFrame(rows).set_index(["arm", "target"]))

oof  oof_delta            oof_ci90  \
arm                      target                                            
+knn_in_category         category  0.7576    -0.0488   [-0.1196, 0.0066]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
+m1_in_category          category  0.7536    -0.0528      [-0.1329, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
+unreviewed_augmentation category  0.8953     0.0889    [0.0273, 0.1739]   
                         priority  0.9265     0.0229       [0.0, 0.0717]   
-augmentation            category  0.8064     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m1                      category  0.8064     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m2knn                   category  0.8064     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m2lr                    category  0.8276     0.0212       [0.0, 0.0601]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m2proto                 category  0.5611    -0.2453   [-0.365, -0.1449]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-urgency_features        category  0.8064     0.0000          [0.0, 0.0]   
                         priority  0.5880    -0.3156  [-0.4648, -0.1785]   
-wrapper_stripping       category  0.7524    -0.0540    [-0.108, 0.0041]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
full                     category  0.8064     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
hard_targets             category  0.8064     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
mask_deadline            category  0.8064     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
proto_learned_only       category  0.5965    -0.2099  [-0.3347, -0.0985]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
proto_office_only        category  0.7609    -0.0455   [-0.1636, 0.0667]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
tuned_stage_b            category  0.8276     0.0212       [0.0, 0.0601]   
                         priority  0.9036     0.0000          [0.0, 0.0]   

                                   our_test  our_test_delta  
arm                      target                              
+knn_in_category         category    0.6976         -0.0910  
                         priority    0.8473          0.0000  
+m1_in_category          category    0.7848         -0.0037  
                         priority    0.8473          0.0000  
+unreviewed_augmentation category    0.8523          0.0637  
                         priority    0.9081          0.0607  
-augmentation            category    0.7886          0.0000  
                         priority    0.8473          0.0000  
-m1                      category    0.7886          0.0000  
                         priority    0.8473          0.0000  
-m2knn                   category    0.7886          0.0000  
                         priority    0.8473          0.0000  
-m2lr                    category    0.7848         -0.0037  
                         priority    0.8473          0.0000  
-m2proto                 category    0.6742         -0.1143  
                         priority    0.8473          0.0000  
-urgency_features        category    0.7886          0.0000  
                         priority    0.5985         -0.2489  
-wrapper_stripping       category    0.7224         -0.0662  
                         priority    0.7806         -0.0667  
full                     category    0.7886          0.0000  
                         pr